In [ ]:
from ultralytics import YOLO
from pytubefix import YouTube
import cv2
import os
import shutil
import torch
import glob

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

In [ ]:
### FOR TRAINING ONLY

model = YOLO("yolo11m.pt")

model.train(
    data="datasets/cctv/data.yaml",
    epochs=100,
    device=0,
    amp=True,
    imgsz=512, # Keept at 512. 640 is too slow
    batch=16, # Keep at 8 or lower
    workers=4, # Keep at 4 or lower
    plots=True # False just in case
)

In [ ]:
### Main Code Area

detect_model = YOLO("models/11m.pt")
pose_model = YOLO("yolo11m-pose.pt")

min_conf = 0.5

detect_results = detect_model.track(
    source="video/input/test_video_3.mp4",
    device=0,
    half=True,
    stream=True,
    tracker="botsort.yaml"
)

frame_count = 0

output_video_count = 0
output_path = "video/output"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
fps = 30
first = True
writer = None

for result in detect_results:
    frame = result.orig_img.copy()
    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        continue

    track_ids = (
        boxes.id.int().cpu().tolist()
        if boxes.id is not None
        else [None] * len(boxes)
    )

    if first:
        height, width = frame.shape[:2]
        writer = cv2.VideoWriter(f"{output_path}/output_video{output_video_count}.mp4", fourcc, fps, (width, height))
        first = False

    for box, cls_id, conf, track_id in zip(
        boxes.xyxy,
        boxes.cls,
        boxes.conf,
        track_ids
    ):
        # Video Frame Generation 
        x1, y1, x2, y2 = map(int, box)
        rgb = (0, 0, 255)
        thickness = 2
        label_offset = 5
        label_scale = 0.6
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            rgb,
            thickness
        )
        cls_name = detect_model.names[int(cls_id)]
        label = f"ID:{track_id} CLS:{cls_name} CONF:{conf:.2f}"
        cv2.putText(
            frame,
            label,
            (int(x1), int(y1 - label_offset)),
            cv2.FONT_HERSHEY_SIMPLEX,
            label_scale,
            rgb,
            thickness
        )

        # Guard Clauses
        not_person = result.names[int(cls_id)] != "person"
        not_min_conf = conf < min_conf
        if not_person or not_min_conf:
            continue

        x1, y1, x2, y2 = map(int, box)
        not_valid_crop = x2 <= x1 or y2 <= y1
        if not_valid_crop:
            continue

        crop = frame[y1:y2, x1:x2]
        empty_crop = crop.size == 0
        if empty_crop:
            continue
        
        # Pose Estimation
        pose_results = pose_model(
            crop,
            verbose=False
        )
        pose_result = pose_results[0]
        annotated_crop = pose_result.plot()
        resized = cv2.resize(
            annotated_crop,
            (x2 - x1, y2 - y1)
        )
        frame[y1:y2, x1:x2] = resized

    
    writer.write(frame)

print(f"Saved video: output_video{output_video_count}.mp4")

if writer is not None:
    writer.release()
        


        


video 1/1 (frame 1/694) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\test_video_3.mp4: 512x480 1 person, 2 phones, 60.2ms
video 1/1 (frame 2/694) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\test_video_3.mp4: 512x480 1 person, 2 phones, 23.0ms
video 1/1 (frame 3/694) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\test_video_3.mp4: 512x480 1 person, 2 phones, 9.7ms
video 1/1 (frame 4/694) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\test_video_3.mp4: 512x480 1 person, 2 phones, 10.0ms
video 1/1 (frame 5/694) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\test_video_3.mp4: 512x480 1 person, 2 phones, 10.6ms
video 1/1 (frame 6/694) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\test_video_3.mp4: 512x480 1 person, 2 phones, 10.8ms
video 1/1 (frame 7/694) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\test_video_3.mp4: 512x480 1 person, 2 phones, 10.2ms
video 